# Loan Approval Prediction — Demo
**COEN 330 — Applied Machine Learning**

This notebook loads the best trained model (Gradient Boosting) and predicts whether a loan application should be **approved or rejected** based on applicant information.

To try a different applicant, edit the values in **Step 2** and re-run the notebook.

## Step 1 — Load the Model

In [ ]:
import sys
import os
from pathlib import Path

import pandas as pd
import joblib

PROJECT_ROOT = Path(os.path.abspath('..')).resolve()
MODELS_DIR   = PROJECT_ROOT / 'models'
MODEL_PATH   = MODELS_DIR / 'GradientBoosting.pkl'

if not MODEL_PATH.exists():
    raise FileNotFoundError(
        f'Model not found at {MODEL_PATH}.\n'
        'Run 03_model_training.ipynb first.'
    )

pipeline = joblib.load(MODEL_PATH)
print(f'Model loaded: {MODEL_PATH.name}')

## Step 2 — Define the Applicant

Edit any values below to try a different applicant.

In [ ]:
applicant = {
    'person_age':                      29,
    'person_gender':                   'male',
    'person_education':                'Bachelor',
    'person_income':                   55000,
    'person_emp_exp':                  4,
    'person_home_ownership':           'RENT',
    'loan_amnt':                       12000,
    'loan_intent':                     'EDUCATION',
    'loan_int_rate':                   11.5,
    'loan_percent_income':             0.22,
    'cb_person_cred_hist_length':      5,
    'credit_score':                    640,
    'previous_loan_defaults_on_file':  'No',
}

# Display the applicant profile as a readable table
labels = {
    'person_age':                     'Age',
    'person_gender':                  'Gender',
    'person_education':               'Education',
    'person_income':                  'Annual Income',
    'person_emp_exp':                 'Employment Experience (yrs)',
    'person_home_ownership':          'Home Ownership',
    'loan_amnt':                      'Loan Amount Requested',
    'loan_intent':                    'Loan Purpose',
    'loan_int_rate':                  'Interest Rate (%)',
    'loan_percent_income':            'Loan as % of Income',
    'cb_person_cred_hist_length':     'Credit History Length (yrs)',
    'credit_score':                   'Credit Score',
    'previous_loan_defaults_on_file': 'Previous Loan Defaults',
}

formatted = {}
for key, label in labels.items():
    val = applicant[key]
    if key == 'person_income':   val = f'${val:,.0f}'
    elif key == 'loan_amnt':     val = f'${val:,.0f}'
    elif key == 'loan_int_rate': val = f'{val}%'
    elif key == 'loan_percent_income': val = f'{val:.0%}'
    formatted[label] = val

profile_df = pd.DataFrame.from_dict(formatted, orient='index', columns=['Value'])
profile_df.index.name = 'Feature'
display(profile_df)

## Step 3 — Predict

In [ ]:
X = pd.DataFrame([applicant])

clf = pipeline.named_steps['clf']
if hasattr(clf, 'predict_proba'):
    prob_approved = pipeline.predict_proba(X)[0, 1]
else:
    score = pipeline.decision_function(X)[0]
    prob_approved = 1 / (1 + 2.718 ** -score)

decision = 'APPROVED ✅' if prob_approved >= 0.5 else 'REJECTED ❌'

print('=' * 45)
print(f'  DECISION   : {decision}')
print(f'  CONFIDENCE : {prob_approved:.1%} probability of approval')
print('=' * 45)

## Step 4 — Try a Second Applicant (High Risk)

For comparison during the presentation — a clearly high-risk applicant.

In [ ]:
high_risk = {
    'person_age':                      22,
    'person_gender':                   'female',
    'person_education':                'High School',
    'person_income':                   18000,
    'person_emp_exp':                  0,
    'person_home_ownership':           'RENT',
    'loan_amnt':                       14000,
    'loan_intent':                     'PERSONAL',
    'loan_int_rate':                   18.5,
    'loan_percent_income':             0.78,
    'cb_person_cred_hist_length':      2,
    'credit_score':                    480,
    'previous_loan_defaults_on_file':  'Yes',
}

X2 = pd.DataFrame([high_risk])
if hasattr(clf, 'predict_proba'):
    prob2 = pipeline.predict_proba(X2)[0, 1]
else:
    score2 = pipeline.decision_function(X2)[0]
    prob2 = 1 / (1 + 2.718 ** -score2)

decision2 = 'APPROVED ✅' if prob2 >= 0.5 else 'REJECTED ❌'

print('High-Risk Applicant:')
print('=' * 45)
print(f'  DECISION   : {decision2}')
print(f'  CONFIDENCE : {prob2:.1%} probability of approval')
print('=' * 45)

## Step 5 — Side-by-Side Comparison

In [ ]:
import matplotlib.pyplot as plt

names  = ['Borderline Applicant', 'High-Risk Applicant']
probs  = [prob_approved, prob2]
colors = ['steelblue' if p >= 0.5 else 'tomato' for p in probs]

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(names, probs, color=colors, edgecolor='white', width=0.4)
ax.axhline(0.5, color='gray', linestyle='--', lw=1, label='Decision boundary (0.5)')
for bar, p in zip(bars, probs):
    ax.text(bar.get_x() + bar.get_width() / 2, p + 0.02,
            f'{p:.1%}', ha='center', fontsize=12, fontweight='bold')
ax.set_ylim(0, 1.1)
ax.set_ylabel('P(Approved)')
ax.set_title('Predicted Approval Probability')
ax.legend()
plt.tight_layout()
plt.savefig('../results/plots/demo_comparison.png')
plt.show()